In [ ]:
import numpy as np 

np.random.seed(3)

In [ ]:
from torchvision import datasets
import numpy as np

# 1. 加载数据 (对应 Keras 的 load_data)
train_dataset = datasets.MNIST(root='./data', train=True, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, download=True)

# 提取原始数据 (uint8 0-255)
train_x = train_dataset.data.numpy()
test_x = test_dataset.data.numpy()

# 提取标签
train_y = train_dataset.targets.numpy()
test_y = test_dataset.targets.numpy()

# 2. Reshape (展开成 1维), 对应: train_features.reshape(60000, 784)
train_x = train_x.reshape(train_x.shape[0], 784)
test_x = test_x.reshape(test_x.shape[0], 784)

# 3. 归一化 (0-255 -> 0-1), 对应: train_features = train_features / 255.0
train_x = train_x.astype(np.float32) / 255.0
test_x = test_x.astype(np.float32) / 255.0

#### Fully Connected Layer (Linear Layer)

In [ ]:
class Linear():
    def __init__(self, in_size, out_size):
        self.weight = np.random.randn(in_size, out_size) * 0.01
        self.bias = np.zeros((1, out_size))
        self.params = [self.weight, self.bias]
        self.grad_weight = None
        self.grad_bias = None
        self.gradInput = None        

    def forward(self, x):
        self.x = x
        self.output = np.dot(x, self.weight) + self.bias
        return self.output

    def backward(self, nextgrad):
        self.grad_weight = np.dot(self.x.T, nextgrad)
        self.grad_bias = np.sum(nextgrad, axis=0)
        self.gradInput = np.dot(nextgrad, self.weight.T)
        return self.gradInput, [self.grad_weight, self.grad_bias]

In [ ]:
class ReLU():
    def __init__(self):
        self.params = []
        self.gradInput = None

    def forward(self, x):
        self.output = np.maximum(x, 0)
        return self.output

    def backward(self, nextgrad):
        self.gradInput = nextgrad.copy()
        self.gradInput[self.output <=0] = 0
        return self.gradInput, []

In [ ]:
class Dropout:
    def __init__(self, p=0.5):
        self.p = p
        self.params = []
        self.gradInput = None
        
    def forward(self, x):
        self.mask = np.random.binomial(1, self.p, size=x.shape)
        self.output = x * self.mask
        return self.output
    
    def backward(self, nextgrad):
        self.gradInput = nextgrad * self.mask
        return self.gradInput, []

In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

class CrossEntropy:
    def __init__(self):
        self.loss_type = "CrossEntropy"

    def forward(self, x, y):
        self.m = y.shape[0]
        self.p = softmax(x)
        cross_entropy = -np.log(self.p[range(self.m), y] + 1e-8)
        loss = np.sum(cross_entropy) / self.m
        return loss
    
    def backward(self, x, y):
        grad = softmax(x)
        grad[range(self.m), y] -= 1
        grad /= self.m
        return grad

# ==========================================
# 1. 优化器类 (SGD)
# ==========================================
import numpy as np

class SGD:
    def __init__(self, params, lr=0.01, momentum=0.9):
        """
        params: 模型参数列表 (List of Lists)
        """
        self.params = params
        self.lr = lr
        self.momentum = momentum
        
        # momentum_buffers: 对应 PyTorch 内部 state['momentum_buffer']
        self.momentum_buffers = []

        # param_grads: 对应 PyTorch 中每个参数身上的 .grad 属性
        self.param_grads = []
        
        # 初始化结构
        for layer_params in self.params:
            self.momentum_buffers.append([np.zeros_like(p) for p in layer_params])
            self.param_grads.append([np.zeros_like(p) for p in layer_params])

    def zero_grad(self):
        """
        对应 PyTorch: optimizer.zero_grad()
        将所有累积的梯度 (param.grad) 清零
        """
        for layer_grads in self.param_grads:
            for grad in layer_grads:
                grad.fill(0.0)

    def step(self, backward_grads):
        """
        backward_grads: 从反向传播传来的新梯度 (通常是倒序的)
        """
        # 使用 zip 将四组数据打包：momentum_buffers (动量缓存), params (模型参数 W, b), param_grads (累积梯度缓存, 即 p.grad), backward_grads (新计算的梯度) -> 需要翻转以匹配正序
        
        iterator = zip(
            self.momentum_buffers, 
            self.params, 
            self.param_grads, 
            reversed(backward_grads)
        )
        
        # 遍历每一层 (Layer Group)
        for layer_buffer, layer_params, layer_grads, layer_new_grads in iterator:
            
            # 遍历层内的每一个参数 (Parameter)，例如 W 和 b
            # param: 参数本身
            # grad:  累积梯度 (Buffer)
            # buf:   动量缓存 (Buffer)
            # new_grad: 本次反向传播算出的新梯度
            for i in range(len(layer_params)):
                param = layer_params[i]
                grad = layer_grads[i]
                buf = layer_buffer[i]
                new_grad = layer_new_grads[i]

                # --- 核心逻辑 ---
                
                # 1. 梯度累积 (Gradient Accumulation)
                grad += new_grad
                
                # 2. 动量更新 (Momentum Update)
                # 对应 PyTorch: buf = momentum * buf + grad
                # 注意：PyTorch 默认把 lr 乘在外面，这里为了简单乘在里面，效果一致
                buf[:] = self.momentum * buf + self.lr * grad
                
                # 3. 参数更新 (Weight Update)
                # 对应 PyTorch: p -= buf
                param -= buf

# ==========================================
# 2. 网络类 (NN)
# ==========================================
class NN:
    def __init__(self):
        self.layers = []
        self.params = [] 
        
    def add_layer(self, layer):
        self.layers.append(layer)
        self.params.append(layer.params)

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    
    def backward(self, grad):
        grads = []
        for layer in reversed(self.layers):
            grad, layer_grads = layer.backward(grad)
            grads.append(layer_grads)
        return grads

# ==========================================
# 3. 训练流程
# ==========================================
def train(net, train_x, train_y, test_x, test_y, epochs, learning_rate):
    
    optimizer = SGD(net.params, lr=learning_rate)
    criterion = CrossEntropy()
    
    for i in range(epochs):
        # 1. Zero Grad (现在这一步是神圣不可或缺的)
        # 如果你注释掉这一行，self.grads 里的值会越来越大，Loss 瞬间爆炸
        optimizer.zero_grad() 
        
        # 2. Forward
        output = net.forward(train_x)
        loss = criterion.forward(output, train_y)
        
        # 3. Backward
        loss_grad = criterion.backward(output, train_y)
        calculated_grads = net.backward(loss_grad)
        
        # 4. Step
        # 我们把“传入梯度”这一步合并到了 step 里
        # step 会把 calculated_grads 加到 optimizer 的内部 buffer 里，然后更新
        optimizer.step(calculated_grads)

        # 打印部分
        if i != 0:
            train_pred = output.argmax(axis=1)
            test_pred = net.forward(test_x).argmax(axis=1)
            train_acc = np.mean(train_pred == train_y)
            test_acc = np.mean(test_pred == test_y)
            print(f"Epoch {i}: Loss = {loss:.4f} | Train Acc = {train_acc:.4f} | Test Acc = {test_acc:.4f}")

    return net

In [ ]:
## input size
input_dim = train_x.shape[1]

## hyperparameters
learning_rate = 0.1
hidden_nodes = 32
output_nodes = 10

## define neural net
nn = NN()
nn.add_layer(Linear(input_dim, hidden_nodes))
# nn.add_layer(Dropout(p=0.5))
nn.add_layer(ReLU())
nn.add_layer(Linear(hidden_nodes, output_nodes))

nn = train(nn, train_x , train_y, test_x, test_y, epochs=1000, learning_rate=learning_rate)